# Shell Mesh Builder

Build a 3D shell-like mesh by sweeping an aperture along a growth trajectory.

Conceptually, this treats the shell as a record of accretionary growth:

- The growth trajectory represents the path followed by the growing aperture through space. This may be a logarithmic spiral, a conical helix, a straight orthocone axis, or another growth path.
- At each point on the trajectory, an elliptical aperture is generated.
- The aperture expands as the shell grows.
- Aperture orientation determines how the aperture is rotated relative to the growth trajectory. In coiled shells this typically follows the spiral angle, while in orthocones it may remain fixed so that the aperture stays perpendicular to the shell axis.
- Axial translation, supplied either by z_path or z_scale, lifts the growth path through space and allows high-spired forms.
- Aperture tilt introduces a simple shear, causing one side of the aperture to sit higher than the other. This gives the whorl profile a slight inclination and helps neighbouring whorls read as overlapping shell growth rather than as a separate coiled tube.
- Consecutive apertures are stitched together into a continuous surface.
- Periodic changes in aperture size create growth ribs.
- Late-stage enlargement of the aperture creates a flared shell lip.
- Optional shell wall thickness creates an outer and inner shell surface.
- A colour value is assigned to each vertex to simulate shell banding and pigmentation patterns.

The shell wall thickness is deliberately simple: each aperture ring is duplicated as a slightly smaller inner ring. The outer rings form the external shell surface, while the inner rings form the inner shell wall. The final aperture is stitched between the outer and inner rings so that the shell opening has a visible rim.

## Geometry and Growth Processes

The shell framework separates geometric growth from biological growth processes.

- The growth trajectory defines where the shell grows.
- Aperture orientation defines how the aperture is rotated at each growth stage.
- Aperture size defines how the shell expands through time.
- Growth phase provides an independent parameter that can be used to drive periodic biological features such as pigmentation bands, growth ribs and other repeating ornamentation.

For many spiral shells, growth trajectory, aperture orientation and growth phase are all derived from the same angular parameter. However, they may vary independently in other shell forms. Orthocones provide an important example, where the shell follows a straight growth axis while pigmentation and ribbing can still vary periodically along the shell.

## Important implementation detail

- The outer shell vertices are stored first.
- Inner shell vertices are appended afterwards.

This keeps the first part of the mesh compatible with the chamber/septa builder, which uses the original outer aperture rings as the edges for internal chamber walls.

## Ribbing controls

- Smaller ribbing_amplitude gives subtler ribs.
- Larger ribbing_amplitude gives stronger raised growth ribs.
- Smaller ribbing_frequency gives broader, more widely spaced ribs.
- Larger ribbing_frequency gives finer, more closely spaced ribs.
- ribbing_mode controls the rib profile. The default sinusoidal mode preserves the original smooth ribbing behaviour. The segmented mode creates broad annular segments separated by narrow grooves for annulated orthocones.
- segment_to_septum_ratio can lock segmented annulations to chamber spacing when chamber septa are enabled in the preset. For example, 1.0 gives one ring per chamber, 2.0 gives two rings per chamber, and 0.5 gives one ring every two chambers.

## Lip controls

- lip_start controls when the final aperture flare begins. For example, 0.88 means the lip starts forming after 88% of growth.
- lip_strength controls how much the final aperture expands.
- lip_power controls how gradually or abruptly the flare develops.

## Wall controls

- wall_enabled controls whether an inner shell surface is generated.
- wall_thickness controls how far the inner surface is inset from the outer surface. Smaller values give thinner shell walls.
- inner_surface_colour_shift can darken or lighten the inner shell surface relative to the outer pigmentation.

In [1]:
import numpy as np

In [ ]:
%run maths.ipynb

In [ ]:
def centreline_tangent(x, y, z_path, idx):
    """
    Estimate the local growth tangent along the shell centreline

    The tangent is computed using neighbouring centreline positions and represents the instantaneous
    growth direction at a particular ring. It forms the basis of the local aperture coordinate frame
    used for curved shells.

    :param x: Centreline X coordinates
    :param y: Centreline Y coordinates
    :param z_path: Centreline Z coordinates
    :param idx: Ring index along the centreline
    :returns: Unit tangent vector at the specified position
    """
    if idx == 0:
        delta = np.array([
            x[1] - x[0],
            y[1] - y[0],
            z_path[1] - z_path[0]
        ])
    elif idx == len(x) - 1:
        delta = np.array([
            x[-1] - x[-2],
            y[-1] - y[-2],
            z_path[-1] - z_path[-2]
        ])
    else:
        delta = np.array([
            x[idx + 1] - x[idx - 1],
            y[idx + 1] - y[idx - 1],
            z_path[idx + 1] - z_path[idx - 1]
        ])

    return normalise_vector(delta, fallback=np.array([1.0, 0.0, 0.0]))

In [ ]:
def aperture_point_from_frame(
    cx,
    cy,
    cz,
    local_x,
    local_z,
    width_axis,
    height_axis,
    aperture_tilt
):
    """
    Transform local aperture coordinates into world coordinates

    The aperture point is constructed within a local frame aligned to the shell growth direction,
    allowing aperture geometry to follow curved centreline paths. Aperture tilt is applied within
    the local frame before conversion to world space.

    :param cx: Aperture centre X coordinate
    :param cy: Aperture centre Y coordinate
    :param cz: Aperture centre Z coordinate
    :param local_x: Local width-axis coordinate
    :param local_z: Local height-axis coordinate
    :param width_axis: Local aperture width direction
    :param height_axis: Local aperture height direction
    :param aperture_tilt: Shear applied across the aperture
    :returns: World-space aperture point (x, y, z)
    """
    centre = np.array([cx, cy, cz])

    point = (
        centre
        + local_x * width_axis
        + (local_z + aperture_tilt * local_x) * height_axis
    )

    return point[0], point[1], point[2]

In [2]:
def growth_fraction(idx, n_spiral):
    """
    Return the normalised growth fraction for a spiral index.

    :param idx: Current growth-path index
    :param n_spiral: Total number of growth-path positions
    :return: Fraction from 0.0 to 1.0
    """
    return idx / (n_spiral - 1)

In [ ]:
def smoothstep(edge0, edge1, value):
    """
    Smoothly interpolate from 0 to 1 between two scalar edges.

    :param edge0: Lower transition edge
    :param edge1: Upper transition edge
    :param value: Value to interpolate
    :return: Smooth interpolation value
    """
    if edge0 == edge1:
        return 1.0 if value >= edge1 else 0.0

    t = np.clip((value - edge0) / (edge1 - edge0), 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def compute_ribbing_factor(
    theta_value,
    enabled=True,
    amplitude=0.08,
    frequency=10,
    mode="sinusoidal",
    groove_width=0.18,
    groove_depth=1.0
):
    """
    Compute the geometric ribbing multiplier for an aperture ring.

    :param theta_value: Growth phase at the current growth point
    :param enabled: Whether ribbing is enabled
    :param amplitude: Strength of aperture modulation
    :param frequency: Frequency of ribs along the growth path
    :param mode: Rib profile to use: "sinusoidal" or "segmented"
    :param groove_width: Fraction of each segmented rib cycle occupied by the groove
    :param groove_depth: Relative depth of segmented grooves
    :return: Multiplicative ribbing factor
    """
    if not enabled:
        return 1.0

    if mode == "sinusoidal":
        return 1 + amplitude * np.sin(frequency * theta_value)

    if mode == "segmented":
        # Convert the continuous growth phase into a repeating 0..1 position
        # within one rib cycle. 0.0 and 1.0 are the groove centre; 0.5 is midway
        # through the raised annular segment.
        cycle_position = ((frequency * theta_value) / (2 * np.pi)) % 1.0

        # Measure how close this point is to the groove centre. Because the cycle
        # wraps, positions near both 0.0 and 1.0 are close to the same groove.
        distance_to_groove = min(cycle_position, 1.0 - cycle_position)

        # Convert the configured groove width into a distance from the groove centre
        # to one edge of the groove. The tiny minimum avoids divide-by-zero issues.
        half_groove_width = max(groove_width / 2.0, 1e-6)

        # Build a smooth groove shape. This is near 1.0 at the groove centre and
        # fades to 0.0 at the groove edge, leaving most of the cycle as a raised band.
        groove_profile = 1.0 - smoothstep(0.0, half_groove_width, distance_to_groove)

        # Return the aperture scale factor. Away from grooves this is approximately
        # 1 + amplitude, creating a raised segment. At grooves, groove_depth subtracts
        # from that raised value, producing the annular constriction.
        return 1 + amplitude * (1.0 - groove_depth * groove_profile)

    raise ValueError(f"Unknown ribbing mode: {mode}")

In [4]:
def compute_lip_factor(idx, n_spiral, enabled=True, lip_start=0.88,
                       lip_strength=0.35, lip_power=3):
    """
    Compute the late-growth aperture flare multiplier.

    :param idx: Current growth-path index
    :param n_spiral: Total number of growth-path positions
    :param enabled: Whether lip flare is enabled
    :param lip_start: Growth fraction at which flare begins
    :param lip_strength: Maximum proportional aperture enlargement
    :param lip_power: Controls gradual versus abrupt flare development
    :return: Multiplicative lip flare factor
    """
    if not enabled:
        return 1.0

    gf = growth_fraction(idx, n_spiral)

    if gf < lip_start:
        return 1.0

    lip_t = (gf - lip_start) / (1 - lip_start)
    return 1 + lip_strength * lip_t**lip_power

In [5]:
def compute_inner_axes(aw, ah, wall_enabled=True, wall_thickness=0.04):
    """
    Compute the inner aperture semi-axes used for shell wall thickness.

    :param aw: Outer aperture semi-width
    :param ah: Outer aperture semi-height
    :param wall_enabled: Whether an inner wall is generated
    :param wall_thickness: Absolute inset from the outer aperture
    :return: Inner aperture semi-width and semi-height
    """
    if not wall_enabled:
        return aw, ah

    inner_aw = max(aw - wall_thickness, aw * 0.25)
    inner_ah = max(ah - wall_thickness, ah * 0.25)

    return inner_aw, inner_ah

In [6]:
def pigmentation_band(theta_value, phi_value, ribbing_factor):
    """
    Compute a simple shell pigmentation value for one vertex.

    :param theta_value: Spiral angle at the current growth point
    :param phi_value: Position around the aperture ring
    :param ribbing_factor: Ribbing multiplier for the current aperture
    :return: Scalar colour/intensity value
    """
    return np.sin(8 * theta_value) + 0.4 * np.sin(3 * phi_value) * ribbing_factor

In [7]:
def aperture_point(cx, cy, cz, local_x, local_z, angle, aperture_tilt):
    """
    Convert local aperture coordinates into world-space shell coordinates.

    :param cx: Aperture centre x coordinate
    :param cy: Aperture centre y coordinate
    :param cz: Aperture centre z coordinate
    :param local_x: Local horizontal aperture coordinate
    :param local_z: Local vertical aperture coordinate
    :param angle: Rotation angle of the aperture in the x/y plane
    :param aperture_tilt: Shear applied to z according to local_x
    :return: World-space x, y, z coordinates
    """
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)

    px = cx + local_x * cos_a
    py = cy + local_x * sin_a
    pz = cz + local_z + aperture_tilt * local_x

    return px, py, pz

In [ ]:
def build_aperture_rings(
    theta, x, y, aperture_width, aperture_height,
    z_scale=0.08,
    z_path=None,
    orientation_path=None,
    growth_phase=None,
    ribbing_phase=None,
    aperture_points=32,
    aperture_tilt=0.25,
    ribbing_enabled=True,
    ribbing_amplitude=0.08,
    ribbing_frequency=10,
    ribbing_mode="sinusoidal",
    ribbing_groove_width=0.18,
    ribbing_groove_depth=1.0,
    lip_enabled=True,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3,
    wall_enabled=True,
    wall_thickness=0.04,
    inner_surface_colour_shift=-0.15,
    use_centreline_frame=False
):
    """
    Build outer and optional inner aperture-ring vertices along the growth path.

    Aperture ring orientation is controlled using either a local coordinate frame perpendicular
    to the growth centreline, and derived from the shell centreline tangent, or by the supplied
    angular orientation path. This allows aperture geometry to follow curved growth paths such
    as orthocones and cyrtocones or log-spiral growth paths.

    Modelling note:

    Earlier versions of the project assumed all shell forms could be generated by rotating aperture
    rings around a growth angle. The introduction of orthocones and cyrtocones revealed a second
    growth architecture in which aperture orientation must follow a centreline tangent rather than
    an angular trajectory.

    This function therefore supports both angular-growth and centreline-growth shell families.

    :param theta: Spiral angle values
    :param x: Spiral x coordinates
    :param y: Spiral y coordinates
    :param aperture_width: Aperture width at each growth point
    :param aperture_height: Aperture height at each growth point
    :param z_scale: Vertical rise factor if z_path is not supplied
    :param z_path: Optional explicit z coordinates for the growth path
    :param orientation_path: Controls rotation of the aperture as it is swept along the growth trajectory
    :param growth_phase: Independent parameter that can be used to drive pigmentation etc
    :param ribbing_phase: Optional independent parameter used to place geometric ribs
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_tilt: Shear applied to incline the aperture
    :param ribbing_enabled: Whether geometric ribbing is enabled
    :param ribbing_amplitude: Strength of ribbing modulation
    :param ribbing_frequency: Rib frequency along the shell
    :param ribbing_mode: Rib profile to use
    :param ribbing_groove_width: Fraction of segmented rib cycle occupied by the groove
    :param ribbing_groove_depth: Relative depth of segmented grooves
    :param lip_enabled: Whether final aperture flare is enabled
    :param lip_start: Growth fraction where lip flare begins
    :param lip_strength: Maximum proportional lip enlargement
    :param lip_power: Shape exponent for lip flare
    :param wall_enabled: Whether inner shell wall vertices are generated
    :param wall_thickness: Inset used for inner shell wall
    :param inner_surface_colour_shift: Colour shift applied to inner wall
    :param use_centreline_frame: Aperture ring orientation control
    :return: Outer vertex arrays and inner vertex arrays
    """
    n_spiral = len(theta)
    phi_values = np.linspace(0, 2*np.pi, aperture_points, endpoint=False)

    X_outer, Y_outer, Z_outer, C_outer = [], [], [], []
    X_inner, Y_inner, Z_inner, C_inner = [], [], [], []

    # Sweep successive aperture rings along the growth trajectory.
    #
    # Each ring is constructed from a local elliptical aperture whose size,
    # orientation and decoration may vary during growth.
    for idx in range(n_spiral):
        cx = x[idx]
        cy = y[idx]
        cz = z_path[idx] if z_path is not None else z_scale * theta[idx]

        phase = growth_phase[idx] if growth_phase is not None else theta[idx]
        rib_phase = ribbing_phase[idx] if ribbing_phase is not None else phase
        angle = orientation_path[idx] if orientation_path is not None else theta[idx]

        # Aperture dimensions represent the semi-major and semi-minor axes
        # of the current growth opening.
        aw = aperture_width[idx] / 2
        ah = aperture_height[idx] / 2

        # Apply geometric growth modifiers.
        #
        # Ribbing introduces periodic modulation during growth while the lip
        # factor enlarges the final aperture region to simulate aperture flare.
        rib_factor = compute_ribbing_factor(
            rib_phase,
            enabled=ribbing_enabled,
            amplitude=ribbing_amplitude,
            frequency=ribbing_frequency,
            mode=ribbing_mode,
            groove_width=ribbing_groove_width,
            groove_depth=ribbing_groove_depth
        )

        lip_factor = compute_lip_factor(
            idx,
            n_spiral,
            enabled=lip_enabled,
            lip_start=lip_start,
            lip_strength=lip_strength,
            lip_power=lip_power
        )

        aw *= rib_factor * lip_factor
        ah *= rib_factor * lip_factor

        # Generate a corresponding inner aperture profile if shell wall
        # thickness is enabled.
        inner_aw, inner_ah = compute_inner_axes(
            aw,
            ah,
            wall_enabled=wall_enabled,
            wall_thickness=wall_thickness
        )

        # Determine how the aperture ring should be oriented.
        #
        # Two growth architectures are supported:
        #
        # - angular orientation (logarithmic spirals, ammonites, nautiloids,
        #   turritellae)
        #
        # - centreline-frame orientation (orthocones and cyrtocones)
        #
        # Centreline-based shells construct a moving local coordinate frame
        # perpendicular to the growth tangent, allowing aperture geometry to
        # follow curved centreline paths.
        if use_centreline_frame:
            tangent = centreline_tangent(x, y, z_path, idx)
            width_axis, height_axis = aperture_frame_from_tangent(tangent)
        else:
            angle = orientation_path[idx] if orientation_path is not None else theta[idx]

        # Generate vertices around the aperture perimeter.
        #
        # Local aperture coordinates are transformed into world coordinates
        # using either the angular-orientation model or the centreline-frame
        # model selected above.
        for p in phi_values:
            band = pigmentation_band(phase, p, rib_factor)

            outer_local_x = aw * np.cos(p)
            outer_local_z = ah * np.sin(p)

            if use_centreline_frame:
                # Centreline-frame shells construct the aperture within the
                # moving local coordinate frame.
                ox, oy, oz = aperture_point_from_frame(
                    cx, cy, cz,
                    outer_local_x,
                    outer_local_z,
                    width_axis,
                    height_axis,
                    aperture_tilt
                )
            else:
                # Spiral shells use the traditional angular orientation model
                # where aperture rotation is driven directly by shell growth.
                ox, oy, oz = aperture_point(
                    cx, cy, cz,
                    outer_local_x,
                    outer_local_z,
                    angle,
                    aperture_tilt
                )

            X_outer.append(ox)
            Y_outer.append(oy)
            Z_outer.append(oz)
            C_outer.append(band)

            if wall_enabled:
                inner_local_x = inner_aw * np.cos(p)
                inner_local_z = inner_ah * np.sin(p)

                if use_centreline_frame:
                    ix, iy, iz = aperture_point_from_frame(
                        cx, cy, cz,
                        inner_local_x,
                        inner_local_z,
                        width_axis,
                        height_axis,
                        aperture_tilt
                    )
                else:
                    ix, iy, iz = aperture_point(
                        cx, cy, cz,
                        inner_local_x,
                        inner_local_z,
                        angle,
                        aperture_tilt
                    )

                X_inner.append(ix)
                Y_inner.append(iy)
                Z_inner.append(iz)
                C_inner.append(band + inner_surface_colour_shift)

    return X_outer, Y_outer, Z_outer, C_outer, X_inner, Y_inner, Z_inner, C_inner

In [ ]:
def stitch_ring_surface(
    I, J, K,
    face_growth_index,
    ring_offset,
    n_spiral,
    n_aperture,
    reverse=False):
    """
    Stitch neighbouring aperture rings into a continuous triangular surface.

    :param I: Triangle index list I
    :param J: Triangle index list J
    :param K: Triangle index list K
    :param face_growth_index: Per-triangle growth sample index along the log spiral
    :param ring_offset: Vertex offset for the first ring
    :param n_spiral: Number of aperture rings
    :param n_aperture: Number of vertices per aperture ring
    :param reverse: Whether to reverse triangle winding
    """
    for s in range(n_spiral - 1):
        for p in range(n_aperture):
            current = ring_offset + s * n_aperture + p
            next_p = ring_offset + s * n_aperture + ((p + 1) % n_aperture)

            current_next = ring_offset + (s + 1) * n_aperture + p
            next_next = ring_offset + (s + 1) * n_aperture + ((p + 1) % n_aperture)

            if not reverse:
                I.extend([current, next_p])
                J.extend([current_next, current_next])
                K.extend([next_p, next_next])
            else:
                I.extend([current, next_p])
                J.extend([next_p, next_next])
                K.extend([current_next, current_next])

            face_growth_index.extend([s + 1, s + 1])

In [10]:
def stitch_wall_rim(
    I, J, K,
    face_growth_index,
    outer_start,
    inner_start,
    n_aperture,
    growth_index,
    reverse=False):
    """
    Stitch an outer aperture ring to an inner aperture ring to form a rim.

    :param I: Triangle index list I
    :param J: Triangle index list J
    :param K: Triangle index list K
    :param face_growth_index: Per-triangle growth sample index along the log spiral
    :param outer_start: Start index of the outer aperture ring
    :param inner_start: Start index of the inner aperture ring
    :param n_aperture: Number of vertices in each aperture ring
    :param growth_index: Growth sample index along the log spiral
    :param reverse: Whether to reverse the triangle winding
    """
    for p in range(n_aperture):
        outer_current = outer_start + p
        outer_next = outer_start + ((p + 1) % n_aperture)

        inner_current = inner_start + p
        inner_next = inner_start + ((p + 1) % n_aperture)

        if not reverse:
            I.extend([outer_current, outer_next])
            J.extend([inner_current, inner_current])
            K.extend([outer_next, inner_next])
        else:
            I.extend([outer_current, outer_next])
            J.extend([outer_next, inner_next])
            K.extend([inner_current, inner_current])

        face_growth_index.extend([growth_index, growth_index])

In [ ]:
def build_shell_mesh(
    theta,
    r,
    x,
    y,
    aperture_width,
    aperture_height,
    z_scale=0.08,
    z_path=None,
    orientation_path=None,
    growth_phase=None,
    ribbing_phase=None,
    aperture_points=32,
    aperture_tilt=0.25,
    ribbing_enabled=True,
    ribbing_amplitude=0.08,
    ribbing_frequency=10,
    ribbing_mode="sinusoidal",
    ribbing_groove_width=0.18,
    ribbing_groove_depth=1.0,
    lip_enabled=True,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3,
    wall_enabled=True,
    wall_thickness=0.04,
    inner_surface_colour_shift=-0.15,
    use_centreline_frame=False
):
    """
    Build a 3D shell mesh by sweeping an aperture along a growth path.

    :param theta: Spiral angle values representing growth progression
    :param r: Spiral radius values; retained for API compatibility
    :param x: Spiral x coordinates
    :param y: Spiral y coordinates
    :param aperture_width: Width of elliptical aperture at each growth point
    :param aperture_height: Height of elliptical aperture at each growth point
    :param z_scale: Vertical rise factor if z_path is not supplied
    :param z_path: Optional explicit z coordinates for the growth path
    :param orientation_path: Controls rotation of the aperture as it is swept along the growth trajectory
    :param growth_phase: Independent parameter that can be used to drive pigmentation etc
    :param ribbing_phase: Optional independent parameter used to place geometric ribs
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_tilt: Shear applied to incline the aperture
    :param ribbing_enabled: Whether to add geometric growth ribs
    :param ribbing_amplitude: Strength of ribbing modulation
    :param ribbing_frequency: Frequency of ribs along the shell
    :param ribbing_mode: Rib profile to use
    :param ribbing_groove_width: Fraction of segmented rib cycle occupied by the groove
    :param ribbing_groove_depth: Relative depth of segmented grooves
    :param lip_enabled: Whether to flare the final aperture
    :param lip_start: Fraction of growth at which lip flare begins
    :param lip_strength: Maximum proportional lip enlargement
    :param lip_power: Shape exponent for lip flare
    :param wall_enabled: Whether to generate an inner shell wall
    :param wall_thickness: Inset used for inner wall thickness
    :param inner_surface_colour_shift: Colour shift for the inner wall
    :param use_centreline_frame: Aperture ring orientation control
    :return: X, Y, Z, C vertex arrays and triangle index arrays I, J, K, face_growth_index
    """
    n_spiral = len(theta)
    n_aperture = aperture_points

    (
        X_outer, Y_outer, Z_outer, C_outer,
        X_inner, Y_inner, Z_inner, C_inner
    ) = build_aperture_rings(
        theta, x, y,
        aperture_width,
        aperture_height,
        z_scale=z_scale,
        z_path=z_path,
        orientation_path=orientation_path,
        growth_phase=growth_phase,
        ribbing_phase=ribbing_phase,
        aperture_points=aperture_points,
        aperture_tilt=aperture_tilt,
        ribbing_enabled=ribbing_enabled,
        ribbing_amplitude=ribbing_amplitude,
        ribbing_frequency=ribbing_frequency,
        ribbing_mode=ribbing_mode,
        ribbing_groove_width=ribbing_groove_width,
        ribbing_groove_depth=ribbing_groove_depth,
        lip_enabled=lip_enabled,
        lip_start=lip_start,
        lip_strength=lip_strength,
        lip_power=lip_power,
        wall_enabled=wall_enabled,
        wall_thickness=wall_thickness,
        inner_surface_colour_shift=inner_surface_colour_shift,
        use_centreline_frame=use_centreline_frame
    )

    if wall_enabled:
        X = np.array(X_outer + X_inner)
        Y = np.array(Y_outer + Y_inner)
        Z = np.array(Z_outer + Z_inner)
        C = np.array(C_outer + C_inner)
    else:
        X = np.array(X_outer)
        Y = np.array(Y_outer)
        Z = np.array(Z_outer)
        C = np.array(C_outer)

    I, J, K, face_growth_index = [], [], [], []

    stitch_ring_surface(
        I, J, K, face_growth_index,
        ring_offset=0,
        n_spiral=n_spiral,
        n_aperture=n_aperture,
        reverse=False
    )

    if wall_enabled:
        inner_offset = n_spiral * n_aperture

        stitch_ring_surface(
            I, J, K, face_growth_index,
            ring_offset=inner_offset,
            n_spiral=n_spiral,
            n_aperture=n_aperture,
            reverse=True
        )

        final_outer_start = (n_spiral - 1) * n_aperture
        final_inner_start = inner_offset + (n_spiral - 1) * n_aperture

        stitch_wall_rim(
            I, J, K, face_growth_index,
            outer_start=final_outer_start,
            inner_start=final_inner_start,
            n_aperture=n_aperture,
            growth_index=n_spiral - 1,
            reverse=False
        )

        initial_outer_start = 0
        initial_inner_start = inner_offset

        stitch_wall_rim(
            I, J, K, face_growth_index,
            outer_start=initial_outer_start,
            inner_start=initial_inner_start,
            n_aperture=n_aperture,
            growth_index=0,
            reverse=True
        )

    return X, Y, Z, C, I, J, K, np.array(face_growth_index)